In [60]:
%load_ext autoreload

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
# from utils import evaluate, ScenarioWiseSampler
import os
from torch.utils.data import Subset
from model import DayCentModel
import random
import matplotlib.pyplot as plt
import os
from data import DayCentDataset

# note that here you have changed random to daycent_10_inputs

In [17]:
INPUT_NPY = "/users/6/mehta423/daycent/data/processed/daycent_10_random_inputs.npy"
OUTPUT_NPY = "/users/6/mehta423/daycent/data/processed/daycent_10_random_outputs.npy"
INIT_COND = "/users/6/mehta423/daycent/data/SAS_KGML_090925/InputData/initial_site_conditions.xlsx"
OUTPUT_DIR = "/users/6/mehta423/daycent/output/"
BATCH_SIZE = 2048
EPOCHS = 100
LR = 1e-2
DEVICE = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

In [18]:
# ----------------------
# Dataloader
# ----------------------
# check dataset
dataset = DayCentDataset(INPUT_NPY, OUTPUT_NPY, INIT_COND, apply_scaling=True)
# 1) Define split boundaries
train_size = 5075 * 7  # 35,525
val_size = 5075 * 2     # 10,150
test_size = 5075        # 5,075
# Total: 50,750

# 2) Create index arrays for each split
train_idx = np.arange(0, train_size)                                    # [0, 35524]
val_idx = np.arange(train_size, train_size + val_size)                 # [35525, 45674]
test_idx = np.arange(train_size + val_size, train_size + val_size + test_size)  # [45675, 50749]
all_idx = np.arange(0, train_size + val_size + test_size)                     # [0, 50749]

# 3) Wrap subsets
train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)
test_ds  = Subset(dataset, test_idx)

# 4) Create loaders
all_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Dataset sizes — total: {len(dataset)}, train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

0
Dataset sizes — total: 50750, train: 35525, val: 10150, test: 5075


In [19]:
# infer input dim
sample = dataset[0]
seq_feat_dim = sample["sequence"].shape[1]  # #features
init_dim = sample["init_cond"].shape[0]
year_dim = sample["year_enc"].shape[0]

print(f"Input feature dim: {seq_feat_dim}, init cond dim: {init_dim}, year enc dim: {year_dim}")


model = DayCentModel(input_dim=seq_feat_dim, init_dim=init_dim, year_dim=year_dim)
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_model.pth")))
model.to(DEVICE)

Input feature dim: 19, init cond dim: 245, year enc dim: 16


DayCentModel(
  (init_proj): Linear(in_features=261, out_features=32, bias=True)
  (daily_proj): Linear(in_features=19, out_features=32, bias=True)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True)
  (somsc_attn): AttentionPooling(
    (attn): Linear(in_features=128, out_features=1, bias=True)
    (proj): Linear(in_features=128, out_features=128, bias=True)
  )
  (yield_attn): AttentionPooling(
    (attn): Linear(in_features=128, out_features=1, bias=True)
    (proj): Linear(in_features=128, out_features=128, bias=True)
  )
  (somsc_head): Linear(in_features=128, out_features=1, bias=True)
  (yield_head): Linear(in_features=128, out_features=1, bias=True)
)

In [20]:
all_preds = []
all_trues = []
all_masks = []
for batch in all_loader:
    batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(batch)

    all_preds.append(outputs['yield_pred'].cpu().numpy())
    all_trues.append(batch['yield'].cpu().numpy())
    all_masks.append(batch['yield_mask'].cpu().numpy())

# Concatenate all batches
pred_flat = np.concatenate(all_preds, axis=0)
true_flat = np.concatenate(all_trues, axis=0)
mask_flat = np.concatenate(all_masks, axis=0)

In [21]:
pred_flat * mask_flat

array([156.24231, 151.44002, 163.32076, ...,   0.     ,   0.     ,
         0.     ], shape=(50750,), dtype=float32)

In [22]:
true_flat

array([161.693, 162.688, 168.64 , ...,   0.   ,   0.   ,   0.   ],
      shape=(50750,), dtype=float32)

In [25]:
class ScenarioWiseSampler:
    def __init__(self, input_path: str, indices):
        data_dict = np.load(input_path, allow_pickle=True).item()
        self.data = data_dict["data"]      # (N, 365, #features)
        self.mapping = data_dict["mapping"][indices]  # (N, 3) => (scenario, point_id, year)
        self.columns = list(data_dict["columns"])

    def get_year_indices(self, sid, pid):
        # get indices for all years for a given scenario id and plot id
        mask = (self.mapping[:, 0] == sid) & (self.mapping[:, 2] == pid)
        return np.where(mask)[0]
    
    def __iter__(self):
        for sid, year, pid in self.mapping:
            yield sid, year, pid

    def get_all_unique_pids(self):
        return np.unique(self.mapping[:, 2])
    
    def get_all_unique_scenarios(self):
        unique_scenarios = np.unique(self.mapping[:, 0])
        print("Unique Scenarios in the sampler:")
        for scenario in unique_scenarios:
            print(scenario)
        return unique_scenarios

    def __len__(self) -> int:
        return len(self.mapping)

all_sampler = ScenarioWiseSampler(INPUT_NPY, all_idx)
train_sampler = ScenarioWiseSampler(INPUT_NPY, train_idx)
test_sampler = ScenarioWiseSampler(INPUT_NPY, test_idx)
val_sampler = ScenarioWiseSampler(INPUT_NPY, val_idx)

In [26]:
sids = all_sampler.get_all_unique_scenarios()
pids = all_sampler.get_all_unique_pids()

Unique Scenarios in the sampler:
scenario_1000
scenario_1099
scenario_1390
scenario_1616
scenario_2034
scenario_2600
scenario_2729
scenario_3503
scenario_3932
scenario_4880


In [27]:
print(len(all_sampler))

50750


In [28]:
PLOTS_DIR = '/users/6/mehta423/daycent/output/experiment1/plots'

In [30]:
for sid in sids:
    if not os.path.exists(os.path.join(PLOTS_DIR, sid)):
        os.makedirs(os.path.join(PLOTS_DIR, sid))
    random_pids = random.sample(list(pids), 5)
    for i, pid in enumerate(random_pids):
        year_indices = all_sampler.get_year_indices(sid,pid)
        pred = (pred_flat * mask_flat)[year_indices]
        true = true_flat[year_indices]

        # Assuming pred and true are numpy arrays (or lists)
        years = np.arange(2000, 2025)

        plt.figure(figsize=(10, 5))
        plt.plot(years, true, marker='o', label='True', linewidth=2)
        plt.plot(years, pred, marker='s', label='Predicted', linewidth=2, linestyle='--')

        plt.title('Predicted vs True Values (2000–2024)')
        plt.xlabel('Year')
        plt.ylabel('Value')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, sid, f'{pid}.png'))
        # plt.show()
        plt.close()

        # if i == 10:
        #     break
